### 7.3.3 错误处理：handle_errors参数

受限于模型能力，大模型输出的内容可能并 不符合格式要求 ，ToolStrategy通过其 handle_errors参数 提供了结构化过程错误处理策略，以下是主要的几种方式及其用途：

1. handle_errors=True： `LangChain默认方式`，`捕获所有异常`，并使用LangChain 内置的、信息明确的 错误消息模板 提示模型重试，确保最终能得到符合预定格式的有效数据。适用于大多数希望自动处理错误的通用场景。

2. handle_errors=False：关闭自动重试机制，任何异常都会 直接抛出 ，会 中断程序 运行。

3. handle_errors="自定义字符串"：捕获所有异常，但使用开发者 预设的固定字符串 作为错误消息。适用于需要统一、友好的用户提示，或进行特定业务引导的场景。

4. `handle_errors=ExceptionType`：仅 捕获指定类型(如ValueError) 或元组中的异常类型并进行重试， 其他异常直接抛出 。适用于需要 精准控制 ，只对特定错误进行重试的场景。

5. `handle_errors=callable`：灵活性最高的方式，使用开发者 自定义的函数来处理异常 ，可根据不同的异常类型返回差异化的提示信息。适用于需要复杂、精细化错误处理的场景。

```python
ToolStrategy(
  self,
  schema: type[SchemaT] | UnionType | dict[str, Any],
  *,
  tool_message_content: str | None = None,
  handle_errors: bool | str | type[Exception] | tuple[type[Exception], ...] | Callable[[Exception], str] = True
)
```

#### 情况1：设置为True/False/固定字符串

设计思路：模型对于单条信息的格式化输出请求，输出了多个工具调用请求。也称为多结构化输出错误。

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
)

In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True
        #handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='532bef35-a3f9-4cd6-b02f-4b5dbd7fe05c'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"email":"zhang3@atguigu.com","name":"张三"}',
                    'call_id': 'call_MKdjJxFzMpEce2ahGpbwJv83',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_0c034c36dedaac10016aba84e2501487d28edc6e7bda703e2f',
                    'status': 'completed'
                },
                {
                    'arguments': '{"date":"2026-07-15","event_name":"公司年会"}',
                    'call_id': 'call_TojuwGXMwY7UtDGAUrhyWjeg',
                    'name': 'EventDetails',
                    'type': 'function_call',
                    'id': 'fc_0c034c36dedaac10016aba84e2502887d2a3a2dae007770c16',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0c034c36dedaac10016aba84e02dac87d2a111538a7491d9d0',
                'created_at': 1790608608.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0c034c36dedaac10016aba84e02dac87d2a111538a7491d9d0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'email': 'zhang3@atguigu.com', 'name': '张三'},
                    'id': 'call_MKdjJxFzMpEce2ahGpbwJv83',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'date': '2026-07-15', 'event_name': '公司年会'},
                    'id': 'call_TojuwGXMwY7UtDGAUrhyWjeg',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4508,
                'output_tokens': 71,
                'total_tokens': 4579,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='a829ca3b-59ce-4236-bb13-95b491d003f7',
            tool_call_id='call_MKdjJxFzMpEce2ahGpbwJv83'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='5d1420fc-6e23-4463-9480-85f7d15702ec',
            tool_call_id='call_TojuwGXMwY7UtDGAUrhyWjeg'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"email":"zhang3@atguigu.com","name":"张三"}',
                    'call_id': 'call_sWzoSTfieIWPW7Xmc7Mzob1r',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_0c034c36dedaac10016aba84e5dd6087d2a401cb1035ee95f0',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0c034c36dedaac10016aba84e4af5c87d29584d09a500c80fb',
                'created_at': 1790608612.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
       

模型接收到错误反馈后，再次生成新的调用请求，直至成功或达到内部的最大重试次数。

ToolStrategy中自定义的tool_message_content控制的是成功后的消息内容，不影响错误消息内容。

2）handle_errors设置为False:

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=False
        #handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

3）handle_errors设置为“请检查输入数据”:

In [4]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        #handle_errors=False
        handle_errors="请检查输入数据"
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='3207dabb-a23a-4be5-b909-477961626f11'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"name":"张三","email":"zhang3@atguigu.com"}',
                    'call_id': 'call_X03eyKldBQrYCL1Dd327sCoF',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_1790608757715_u0ymumdkgep',
                    'status': 'completed'
                },
                {
                    'arguments': '{"event_name":"公司年会","date":"2026-07-15"}',
                    'call_id': 'call_kuPsNkL2KWSqxrzqIVYarCeF',
                    'name': 'EventDetails',
                    'type': 'function_call',
                    'id': 'fc_1790608757715_rhxznv3w7qb',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'gen_01M3M9JKH0V51VGX8A38XP61YQ',
                'created_at': 1790608757.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='gen_01M3M9JKH0V51VGX8A38XP61YQ',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_X03eyKldBQrYCL1Dd327sCoF',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_kuPsNkL2KWSqxrzqIVYarCeF',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 126,
                'output_tokens': 71,
                'total_tokens': 197,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='请检查输入数据',
            name='ContactInfo',
            id='7ee44f4b-9a30-42e8-a401-307409e8adca',
            tool_call_id='call_X03eyKldBQrYCL1Dd327sCoF'
        ),
        ToolMessage(
            content='请检查输入数据',
            name='EventDetails',
            id='c2396350-4fd6-45dc-b0cc-0840efcd0868',
            tool_call_id='call_kuPsNkL2KWSqxrzqIVYarCeF'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_08633b7e0bf440b3006aba857743d887d18fe6d2c0089dd8c3',
                    'summary': [],
                    'type': 'reasoning',
                    'encrypted_content': 
'gAAAAABquoV4whmI6CBX-IFETuwyk3Vkm3WX2PSvOi3aH9kydKDGmdU3j_8j2scVFnYkdupqpJwIxSXeF-1lYQrA9XiWrANxfaYnJb4892kW-qiGuT
Ss7xBZQiMAxsLX7uwJL5Ab48ihjWlCv-iWN2DFogSXH7FydGKf58LO8o-7_8mORJN1L8nL7rJg1UFMEjO_j6Tsh5M5nWJX-TE-BB5s4WCpj3aSDCDps
umobSQylAZNZtDvakJo9rf6H-_fahz0EMWYELLYFh_HDDHxPbVD3NRxPclQcqum_aw6WBX2w0SfL_q2pWlhJihZtWuw5dwA8vOm5ieD36gfNPZpqssb
5nAwvSTmGTrwa0ZoU8ONH6ZuLnyjAvcPro2OVHYpiZdGzh25GbDEZJDxAsAUWmB4_awblYsfuP8Cg_97L7rveQYL89_shgZgIYShBOIIeqKNKHp2esH
cvBqrB9EuUIKURCPJsgX6fM1nsJyGP8kSw-5qIsXXhrjZz5wjViUheeLSSzyF1h07LNNgIwUyLjBzg5tZk_HHy_7N8Yr_C3MlxJBltKHffs35HoKApo
5R7LOZ_Vsp1BJRmzgRxniTvjiw1QwUtl98EdJWIGNUv7z5D2N11kw_-5Lc2CjoiIc27N782SvrKfAwMSy0i_LJwgDnwmX_sDK46Lyih56ytUZn6HZBa
05WOr7tXZmhzR6CARkLUP-BIGzPk9E103dWkXm2h8_8Y0hJGrKGGoPGSVhLNSzG3pZuyXVQvm7O-AyFQk9d31x5gKhUQEEwWAI59G7-pXa-4pLFjiYf
vZY5-W2rfAuDPN-KPpPujItTYzcCAObn8D5lhPvyJw_piL9aSLAsJqdGdNR231g-C

以上代码中注意如下几点：
1. ToolStrategy允许指定多个类型“Union[ContactInfo, EventDetails]”这种写法，但是最终只会转换成一种结构化类型输出。
2. 当ToolStrategy通过“Union[ContactInfo, EventDetails]”指定多个类型时，在内部调用生成结构化类型工具会报错。此时:
    * handle_errors=True（默认值）开始发挥作用，系统会生成一个ToolMessage，明确告诉LLM“Error: Model incorrectly returned multiple structured responses (ContactInfo,EventDetails) when only one is expected.”，大模型收到这个精准的反馈后，会重新进行推理，最终选择并输出一个最符合要求的Schema。
    * 如果handle_errors 设置为False，执行代码过程直接报错。
3. 当格式化输出有错误时，Agent内部会进行工具调用重试，直到符合要求格式化输出前， 可能会进行多次重试 。

#### 情况2：设置为指定异常类型
默认情况下，LangChain会处理结构化输出处理时抛出的两类异常：
1. MultipleStructuredOutputsError
    
    多结构化输出错误，当返回的工具调用请求数量大于1时，抛出该异常，默认情况下LangChain会拦截异常并提醒模型重试。

2. StructuredOutputValidationError

    输出结构化验证错误，当输出格式不符合结构化要求时，抛出上述异常。默认情况下LangChain会拦截该异常并自动重试。

In [5]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import MultipleStructuredOutputsError, ToolStrategy,StructuredOutputValidationError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        #handle_errors=False
        handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='9939e62a-0999-4421-8796-321df134445e'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"email":"zhang3@atguigu.com","name":"张三"}',
                    'call_id': 'call_B3ffvGQ090IBqQePo7idarNe',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_0b154de4db362636016aba890b5e2887d2bbc82b16bc7354b7',
                    'status': 'completed'
                },
                {
                    'arguments': '{"date":"2026-07-15","event_name":"公司年会"}',
                    'call_id': 'call_3zuB5TjoeEjSUMyMpevJaKZo',
                    'name': 'EventDetails',
                    'type': 'function_call',
                    'id': 'fc_0b154de4db362636016aba890b5e4887d2809f0e0e211bf06b',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0b154de4db362636016aba8909658c87d292a1ca2d6c4b117d',
                'created_at': 1790609673.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0b154de4db362636016aba8909658c87d292a1ca2d6c4b117d',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'email': 'zhang3@atguigu.com', 'name': '张三'},
                    'id': 'call_B3ffvGQ090IBqQePo7idarNe',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'date': '2026-07-15', 'event_name': '公司年会'},
                    'id': 'call_3zuB5TjoeEjSUMyMpevJaKZo',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4508,
                'output_tokens': 71,
                'total_tokens': 4579,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='fdc994ec-df86-490b-b898-3879f11af9be',
            tool_call_id='call_B3ffvGQ090IBqQePo7idarNe'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='d66d1224-bf69-4851-afda-acec6cb35e28',
            tool_call_id='call_3zuB5TjoeEjSUMyMpevJaKZo'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"email":"zhang3@atguigu.com","name":"张三"}',
                    'call_id': 'call_2sIcjRPUezbuQQcCqbeWOBNg',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_0b154de4db362636016aba891a209887d2b0e964198465e035',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0b154de4db362636016aba89191e8487d281f77dff5addd94e',
                'created_at': 1790609689.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
       

In [7]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import MultipleStructuredOutputsError, ToolStrategy,StructuredOutputValidationError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        #handle_errors=False
        handle_errors=(StructuredOutputValidationError)#如果不是StructuredOutputValidationError类型的错误，则会抛出异常
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

#### 情况3：设置为自定义错误处理函数
指定异常处理函数并返回字符串时，**LangChain会在遇到异常时自动重试,并将异常处理函数的返回值作为ToolMessage的内容。**

我们也可以选择在异常处理函数中抛出异常。


In [8]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy,StructuredOutputValidationError, MultipleStructuredOutputsError
from rich import print as rprint
# 自定义错误处理函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到错误类型：{type(error).__name__}")
    print(f"错误详情：{error_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回。"
    else:
        return f"Error: {error_str}"

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
        response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        #handle_errors=False
        handle_errors=custom_error_handler
    )
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(response)

捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='332788ea-2ac9-4a18-8a01-5ce04cc83157'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"name":"张三","email":"zhang3@atguigu.com"}',
                    'call_id': 'call_dqDvNt8DOXN5qm0xNiFLoexE',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_1790610444419_w85bj4g8vpg',
                    'status': 'completed'
                },
                {
                    'arguments': '{"event_name":"公司年会","date":"2026-07-15"}',
                    'call_id': 'call_d0RVpzBAcHMuLyyqTRmS5RhT',
                    'name': 'EventDetails',
                    'type': 'function_call',
                    'id': 'fc_1790610444419_fw72rn3ty49',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'gen_01M3MB62AHEK1KR1BGGPY4H294',
                'created_at': 1790610444.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='gen_01M3MB62AHEK1KR1BGGPY4H294',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_dqDvNt8DOXN5qm0xNiFLoexE',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_d0RVpzBAcHMuLyyqTRmS5RhT',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 126,
                'output_tokens': 71,
                'total_tokens': 197,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='ContactInfo',
            id='cac10fb1-09cf-43e6-80a8-1bab0866e42c',
            tool_call_id='call_dqDvNt8DOXN5qm0xNiFLoexE'
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='EventDetails',
            id='c9c9634e-48d7-4d6f-9296-4ac08a564946',
            tool_call_id='call_d0RVpzBAcHMuLyyqTRmS5RhT'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_0b0f622df856cda8006aba8c0e4ffc87d1975b20d826372efb',
                    'summary': [],
                    'type': 'reasoning',
                    'encrypted_content': 
'gAAAAABquowQKxJ9skA0KU2xNvYTcPJXBLB3pWa35cf8OEECm-cyoyp5GXAkfqavIx9R6FN5G-tYGOXOJ9nYSXxlmNm8fmQRiKsf99CtyQ0a84KHhv
JclYyhlz0WeaGR-yepii7h1M1dQ39Mhr0WKTR4a0vmsHscEQoh6vxIK3b-s8oK8PtLTA1uhVtWQtMdDPgzc2s1DAJ4i2cUciW09WzuCMDTPOalrkZb_
f_t-_xibjhjfBzhcBjuK5mLFtzc36OI4NO2yb6IZVz_VgvWlxxMWntimsa9PJeTmxBJDEYKjWn6-RLZDIpbu8BWwN-wnpLpVLr1hf1U2g5c--bPpXd0
8dsegxXORtUcRE51yKlWiRjScrBD-8Wdo0ZwAdUc7hN510s8AzIkxbTRaaDh3Cv6dt8KqN3ZjPCwsfQeb1NFoy8kaokAq49ZP--oYIKu6ep1N6Sm9Hy
S40dhO7FgUkrlYk1V8CBNDnI16z2805bN7g8aP4CyTIbS6nkhj3imr-LYkZc2saGUH7CLvAtRxjjZNkbJY9-Nybq65p_zR8ek73kQSm8xzZkVfU-2Sa
B7AOA_XQDR7Hq4nX1qytmqIjU9MW4R1f89yx4KF90GPSZT3xB1akQwjDYjzGGUrE06iaW7GUZtbM5MnfX32Zv7Swncpwr4IFVNK6vPLby_gsxGD-5LO
76kvPXhw2kKTSwPvexVoxe27O1Mt2GWKlDZ0GQzh8IugZ_Xj9VzDQbohq1JraRq6mUzVz47m1R8snz8_SuwjoqgsKZJVlevLxgGV1dP4iKQ6yCeOef8
CU1gSPhtMlgfFy8EMWCURr-FnGAax2nwevU